# 03 - Baseline con Feature Preaddestrate (SVM)

In questo notebook stabiliamo una "Baseline Stabile e Riproducibile".
Estraiamo le feature da una ResNet18 (congelata) e le passiamo a un classificatore SVM lineare.
Tutta la logica di inizializzazione è demandata ai moduli del package `src`.

In [1]:
import sys
if '..' not in sys.path:
    sys.path.append('..')

import torch
import time
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

from src.dataset import get_dataloaders
from src.models import get_resnet18_feature_extractor
from src.utils import extract_features, set_seed

# Impostazione per la riproducibilità totale
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device selezionato: {device}")

Device selezionato: cuda


## 1. Caricamento Dati e Modello

Inizializziamo i DataLoader e la ResNet18 modificata per restituire direttamente il vettore a 512 dimensioni.

In [2]:
if __name__ == '__main__':
    # Caricamento Dati
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir="../_data",
        batch_size=128,
        num_workers=8,
        device=device,
        normalization="imagenet",
    )
    
    # Caricamento Modello
    model = get_resnet18_feature_extractor(device=device)
    print("Dati e Modello caricati correttamente.")

Dati e Modello caricati correttamente.


## 2. Estrazione delle Feature

Passiamo le immagini nella rete per estrarre le feature e salvarle su CPU, pronte per essere date in pasto al classificatore SVM.

In [3]:
if __name__ == '__main__':
    print("Inizio estrazione feature per il Training Set...")
    train_feats, train_classes = extract_features(model, train_loader, device, desc="Train Features")
    
    print("\nInizio estrazione feature per il test set...")
    test_feats, test_classes = extract_features(model, test_loader, device, desc="Test Features")
    
    print(f"\nDimensione feature di train: {train_feats.shape}")
    print(f"Dimensione feature di test: {test_feats.shape}")

Inizio estrazione feature per il Training Set...


Train Features: 100%|██████████| 167/167 [00:55<00:00,  3.01it/s]



Inizio estrazione feature per il test set...


Test Features: 100%|██████████| 99/99 [00:34<00:00,  2.86it/s]


Dimensione feature di train: torch.Size([21312, 512])
Dimensione feature di test: torch.Size([12630, 512])


## 3. Classificazione con SVM

Addestriamo la SVM lineare sui vettori delle feature estratte.

In [4]:
if __name__ == '__main__':
    print("Addestramento SVM lineare in corso...")
    start_time = time.time()
    
    svc = SVC(kernel='linear')
    svc.fit(train_feats.numpy(), train_classes.numpy())
    
    print(f"Addestramento completato in {time.time() - start_time:.2f} secondi.")

Addestramento SVM lineare in corso...
Addestramento completato in 10.18 secondi.


## 4. Valutazione

Generiamo le predizioni sul test set e valutiamo i risultati con le metriche standard.

In [5]:
if __name__ == '__main__':
    print("Generazione predizioni...")
    preds = svc.predict(test_feats.numpy())
    trues = test_classes.numpy()
    
    print("\n--- CLASSIFICATION REPORT ---")
    print(classification_report(trues, preds))
    macro_f1 = f1_score(trues, preds, average='macro')
    print(f"Macro F1-Score: {macro_f1:.4f}")

Generazione predizioni...

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.29      0.32      0.30        60
           1       0.46      0.58      0.51       720
           2       0.43      0.46      0.44       750
           3       0.32      0.35      0.34       450
           4       0.53      0.52      0.53       660
           5       0.54      0.49      0.52       630
           6       0.98      0.79      0.88       150
           7       0.57      0.50      0.53       450
           8       0.48      0.42      0.45       450
           9       0.89      0.74      0.81       480
          10       0.82      0.90      0.86       660
          11       0.44      0.53      0.48       420
          12       0.94      0.97      0.95       690
          13       0.95      0.99      0.97       720
          14       0.93      0.84      0.88       270
          15       0.99      0.94      0.97       210
          16       0.88 

## 5. Analisi dei risultati

- La baseline ottiene test accuracy `0,62` e F1 macro `0,5311`.
- Il divario tra accuracy e F1 macro conferma l'effetto dello sbilanciamento: le classi frequenti incidono maggiormente sull'accuracy, mentre diverse classi rare mostrano recall e F1 bassi.
- Alcune categorie visivamente distintive raggiungono F1 elevati, mentre segnali simili o poco rappresentati restano difficili per un classificatore lineare.